# Gold Layer Validation and Reconciliation

Validate the completed UrbanPulse Gold dimensional model, fact tables, and serving layer.

This notebook checks:

1. Required Gold tables exist.
2. Core tables contain data.
3. Dimension and fact keys are unique.
4. SCD Type 2 dimensions are internally consistent.
5. Facts reconcile to Silver.
6. Fact foreign keys resolve correctly.
7. Serving-table grains are valid.
8. Serving tables reconcile to their Gold sources.
9. Validation results are persisted for audit and observability.

In [0]:
# 1. Initialise project paths
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

In [0]:
# 2. Imports
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# 3. Define Tables
SILVER_LINE_STATUS = (
    "workspace.urbanpulse_silver.tfl_line_status"
)

SILVER_ARRIVALS = (
    "workspace.urbanpulse_silver.tfl_arrivals"
)

SILVER_WEATHER = (
    "workspace.urbanpulse_silver.weather"
)


DIM_DATE = (
    "workspace.urbanpulse_gold.dim_date"
)

DIM_LINE = (
    "workspace.urbanpulse_gold.dim_line"
)

DIM_STATION = (
    "workspace.urbanpulse_gold.dim_station"
)

BRIDGE_STATION_LINE = (
    "workspace.urbanpulse_gold.bridge_station_line"
)

FACT_LINE_STATUS = (
    "workspace.urbanpulse_gold.fact_line_status"
)

FACT_ARRIVAL = (
    "workspace.urbanpulse_gold.fact_arrival_observation"
)

FACT_WEATHER = (
    "workspace.urbanpulse_gold.fact_weather"
)

CURRENT_LINE_STATUS = (
    "workspace.urbanpulse_gold.current_line_status"
)

STATION_ARRIVAL_SUMMARY = (
    "workspace.urbanpulse_gold.station_arrival_summary"
)

DAILY_NETWORK_KPIS = (
    "workspace.urbanpulse_gold.daily_network_kpis"
)


VALIDATION_TABLE = (
    "workspace."
    "urbanpulse_meta."
    "gold_validation_results"
)

In [0]:
# 4. Create validation framework
validation_run_id = str(uuid4())
validation_results = []


def record_check(
    check_name,
    status,
    actual=None,
    expected=None,
    details=None,
):
    validation_results.append({
        "validation_run_id":
            validation_run_id,

        "check_name":
            check_name,

        "status":
            status,

        "actual_value":
            None if actual is None
            else str(actual),

        "expected_value":
            None if expected is None
            else str(expected),

        "details":
            details,

        "checked_at":
            datetime.now(timezone.utc),
    })


def check_equal(
    check_name,
    actual,
    expected,
    details=None,
):
    status = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    record_check(
        check_name,
        status,
        actual,
        expected,
        details,
    )


def check_zero(
    check_name,
    actual,
    details=None,
):
    check_equal(
        check_name,
        actual,
        0,
        details,
    )


def check_positive(
    check_name,
    actual,
    details=None,
):
    status = (
        "PASS"
        if actual > 0
        else "FAIL"
    )

    record_check(
        check_name,
        status,
        actual,
        "> 0",
        details,
    )

## 1. Validate required Gold tables

Every expected dimension, fact, bridge, and serving table must exist.

In [0]:
required_tables = [
    DIM_DATE,
    DIM_LINE,
    DIM_STATION,
    BRIDGE_STATION_LINE,
    FACT_LINE_STATUS,
    FACT_ARRIVAL,
    FACT_WEATHER,
    CURRENT_LINE_STATUS,
    STATION_ARRIVAL_SUMMARY,
    DAILY_NETWORK_KPIS,
]

for table_name in required_tables:

    exists = (
        spark.catalog.tableExists(
            table_name
        )
    )

    check_equal(
        f"table_exists::{table_name}",
        exists,
        True,
    )

In [0]:
# Validate core tables contain data
for table_name in required_tables:

    if spark.catalog.tableExists(
        table_name
    ):

        row_count = (
            spark.table(
                table_name
            ).count()
        )

        check_positive(
            f"table_non_empty::{table_name}",
            row_count,
        )

In [0]:
# Validate dim-date
dim_date_df = spark.table(
    DIM_DATE
)

duplicate_date_keys = (
    dim_date_df
    .groupBy("date_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_dates = (
    dim_date_df
    .groupBy("calendar_date")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

check_zero(
    "dim_date_duplicate_date_keys",
    duplicate_date_keys,
)

check_zero(
    "dim_date_duplicate_calendar_dates",
    duplicate_dates,
)

In [0]:
# Validate dim_line SCD type 2
dim_line_df = spark.table(
    DIM_LINE
)

duplicate_line_keys = (
    dim_line_df
    .groupBy("line_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_current_lines = (
    dim_line_df
    .filter(
        F.col("is_current")
    )
    .groupBy("line_id")
    .count()
    .filter(
        F.col("count") != 1
    )
    .count()
)

invalid_line_dates = (
    dim_line_df
    .filter(
        F.col("effective_to").isNotNull()
        &
        (
            F.col("effective_to")
            <=
            F.col("effective_from")
        )
    )
    .count()
)

invalid_line_current_end = (
    dim_line_df
    .filter(
        F.col("is_current")
        &
        F.col("effective_to").isNotNull()
    )
    .count()
)

check_zero(
    "dim_line_duplicate_line_keys",
    duplicate_line_keys,
)

check_zero(
    "dim_line_invalid_current_versions",
    invalid_current_lines,
)

check_zero(
    "dim_line_invalid_effective_dates",
    invalid_line_dates,
)

check_zero(
    "dim_line_current_rows_with_end_date",
    invalid_line_current_end,
)

In [0]:
# Check dim_line for overlapping SCD versions
line_window = (
    Window
    .partitionBy("line_id")
    .orderBy("effective_from")
)

line_overlap_df = (
    dim_line_df
    .withColumn(
        "previous_effective_to",
        F.lag(
            "effective_to"
        ).over(line_window),
    )
    .filter(
        F.col(
            "previous_effective_to"
        ).isNotNull()
        &
        (
            F.col("effective_from")
            <
            F.col(
                "previous_effective_to"
            )
        )
    )
)

check_zero(
    "dim_line_overlapping_versions",
    line_overlap_df.count(),
)

In [0]:
# Validate dim_station
dim_station_df = spark.table(
    DIM_STATION
)

duplicate_station_keys = (
    dim_station_df
    .groupBy("station_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_current_stations = (
    dim_station_df
    .filter(
        F.col("is_current")
    )
    .groupBy("station_id")
    .count()
    .filter(
        F.col("count") != 1
    )
    .count()
)

invalid_station_dates = (
    dim_station_df
    .filter(
        F.col("effective_to").isNotNull()
        &
        (
            F.col("effective_to")
            <=
            F.col("effective_from")
        )
    )
    .count()
)

check_zero(
    "dim_station_duplicate_station_keys",
    duplicate_station_keys,
)

check_zero(
    "dim_station_invalid_current_versions",
    invalid_current_stations,
)

check_zero(
    "dim_station_invalid_effective_dates",
    invalid_station_dates,
)

In [0]:
# check station SCD overlaps
station_window = (
    Window
    .partitionBy("station_id")
    .orderBy("effective_from")
)

station_overlap_df = (
    dim_station_df
    .withColumn(
        "previous_effective_to",
        F.lag(
            "effective_to"
        ).over(station_window),
    )
    .filter(
        F.col(
            "previous_effective_to"
        ).isNotNull()
        &
        (
            F.col("effective_from")
            <
            F.col(
                "previous_effective_to"
            )
        )
    )
)

check_zero(
    "dim_station_overlapping_versions",
    station_overlap_df.count(),
)

In [0]:
# Validate bridge
bridge_df = spark.table(
    BRIDGE_STATION_LINE
)

duplicate_bridge_keys = (
    bridge_df
    .groupBy("station_line_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_relationships = (
    bridge_df
    .groupBy(
        "station_id",
        "line_id",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

check_zero(
    "bridge_duplicate_keys",
    duplicate_bridge_keys,
)

check_zero(
    "bridge_duplicate_relationships",
    duplicate_relationships,
)

In [0]:
# Bridge referential integrity
orphan_bridge_stations = (
    bridge_df
    .join(
        dim_station_df.select(
            "station_key"
        ),
        on="station_key",
        how="left_anti",
    )
    .count()
)

orphan_bridge_lines = (
    bridge_df
    .join(
        dim_line_df.select(
            "line_key"
        ),
        on="line_key",
        how="left_anti",
    )
    .count()
)

check_zero(
    "bridge_orphan_station_keys",
    orphan_bridge_stations,
)

check_zero(
    "bridge_orphan_line_keys",
    orphan_bridge_lines,
)

In [0]:
# Reconcile fact_line_status to Silver
silver_line_status_df = spark.table(
    SILVER_LINE_STATUS
)

fact_line_status_df = spark.table(
    FACT_LINE_STATUS
)

silver_line_count = (
    silver_line_status_df.count()
)

gold_line_count = (
    fact_line_status_df.count()
)

check_equal(
    "fact_line_status_row_reconciliation",
    gold_line_count,
    silver_line_count,
)

In [0]:
duplicate_line_fact_keys = (
    fact_line_status_df
    .groupBy("line_status_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

orphan_line_fact_dimension = (
    fact_line_status_df
    .join(
        dim_line_df.select(
            "line_key"
        ),
        on="line_key",
        how="left_anti",
    )
    .count()
)

orphan_line_fact_date = (
    fact_line_status_df
    .join(
        dim_date_df.select(
            F.col(
                "date_key"
            ).alias(
                "snapshot_date_key"
            )
        ),
        on="snapshot_date_key",
        how="left_anti",
    )
    .count()
)

check_zero(
    "fact_line_status_duplicate_keys",
    duplicate_line_fact_keys,
)

check_zero(
    "fact_line_status_orphan_line_keys",
    orphan_line_fact_dimension,
)

check_zero(
    "fact_line_status_orphan_date_keys",
    orphan_line_fact_date,
)

In [0]:
# Reconcile arrival facts
silver_arrival_df = spark.table(
    SILVER_ARRIVALS
)

fact_arrival_df = spark.table(
    FACT_ARRIVAL
)

silver_arrival_count = (
    silver_arrival_df.count()
)

gold_arrival_count = (
    fact_arrival_df.count()
)

check_equal(
    "fact_arrival_row_reconciliation",
    gold_arrival_count,
    silver_arrival_count,
)

In [0]:
duplicate_arrival_keys = (
    fact_arrival_df
    .groupBy(
        "arrival_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

orphan_arrival_stations = (
    fact_arrival_df
    .join(
        dim_station_df.select(
            "station_key"
        ),
        on="station_key",
        how="left_anti",
    )
    .count()
)

orphan_arrival_lines = (
    fact_arrival_df
    .join(
        dim_line_df.select(
            "line_key"
        ),
        on="line_key",
        how="left_anti",
    )
    .count()
)

check_zero(
    "fact_arrival_duplicate_keys",
    duplicate_arrival_keys,
)

check_zero(
    "fact_arrival_orphan_station_keys",
    orphan_arrival_stations,
)

check_zero(
    "fact_arrival_orphan_line_keys",
    orphan_arrival_lines,
)

In [0]:
# Reconcile weather facts
silver_weather_df = spark.table(
    SILVER_WEATHER
)

fact_weather_df = spark.table(
    FACT_WEATHER
)

check_equal(
    "fact_weather_row_reconciliation",
    fact_weather_df.count(),
    silver_weather_df.count(),
)

In [0]:
duplicate_weather_keys = (
    fact_weather_df
    .groupBy(
        "weather_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

check_zero(
    "fact_weather_duplicate_keys",
    duplicate_weather_keys,
)

In [0]:
# Reconcile current_line_status
current_line_status_df = spark.table(
    CURRENT_LINE_STATUS
)

current_dim_line_count = (
    dim_line_df
    .filter(
        F.col("is_current")
    )
    .count()
)

check_equal(
    "current_line_status_line_coverage",
    current_line_status_df.count(),
    current_dim_line_count,
)

In [0]:
duplicate_current_lines = (
    current_line_status_df
    .groupBy("line_id")
    .count()
    .filter(
        F.col("count") != 1
    )
    .count()
)

check_zero(
    "current_line_status_duplicate_lines",
    duplicate_current_lines,
)

In [0]:
# Validate station_arrival_summary
station_summary_df = spark.table(
    STATION_ARRIVAL_SUMMARY
)

duplicate_station_summaries = (
    station_summary_df
    .groupBy(
        "station_key",
        "line_key",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_eta_summaries = (
    station_summary_df
    .filter(
        (
            F.col("min_eta_seconds")
            >
            F.col("avg_eta_seconds")
        )
        |
        (
            F.col("avg_eta_seconds")
            >
            F.col("max_eta_seconds")
        )
    )
    .count()
)

check_zero(
    "station_summary_duplicate_grain",
    duplicate_station_summaries,
)

check_zero(
    "station_summary_invalid_eta_ranges",
    invalid_eta_summaries,
)

In [0]:
# Validate daily_network_kpis
daily_kpi_df = spark.table(
    DAILY_NETWORK_KPIS
)

duplicate_kpi_dates = (
    daily_kpi_df
    .groupBy("date_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_line_totals = (
    daily_kpi_df
    .filter(
        F.col("line_snapshots")
        !=
        (
            F.col(
                "good_service_line_snapshots"
            )
            +
            F.col(
                "disrupted_line_snapshots"
            )
        )
    )
    .count()
)

check_zero(
    "daily_kpi_duplicate_dates",
    duplicate_kpi_dates,
)

check_zero(
    "daily_kpi_line_totals",
    invalid_line_totals,
)

In [0]:
# Reconcile KPI date coverage
expected_dates_df = (
    fact_line_status_df
    .select(
        F.col(
            "snapshot_date_key"
        ).alias("date_key")
    )
    .unionByName(
        fact_arrival_df.select(
            F.col(
                "prediction_date_key"
            ).alias("date_key")
        )
    )
    .unionByName(
        fact_weather_df.select(
            F.col(
                "observation_date_key"
            ).alias("date_key")
        )
    )
    .distinct()
)

actual_dates_df = (
    daily_kpi_df
    .select("date_key")
    .distinct()
)

missing_kpi_dates = (
    expected_dates_df
    .join(
        actual_dates_df,
        on="date_key",
        how="left_anti",
    )
    .count()
)

unexpected_kpi_dates = (
    actual_dates_df
    .join(
        expected_dates_df,
        on="date_key",
        how="left_anti",
    )
    .count()
)

check_zero(
    "daily_kpi_missing_dates",
    missing_kpi_dates,
)

check_zero(
    "daily_kpi_unexpected_dates",
    unexpected_kpi_dates,
)

In [0]:
# Build validation result dataset
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
)


validation_schema = StructType([
    StructField(
        "validation_run_id",
        StringType(),
        False,
    ),
    StructField(
        "check_name",
        StringType(),
        False,
    ),
    StructField(
        "status",
        StringType(),
        False,
    ),
    StructField(
        "actual_value",
        StringType(),
        True,
    ),
    StructField(
        "expected_value",
        StringType(),
        True,
    ),
    StructField(
        "details",
        StringType(),
        True,
    ),
    StructField(
        "checked_at",
        TimestampType(),
        False,
    ),
])


validation_df = spark.createDataFrame(
    validation_results,
    schema=validation_schema,
)


display(
    validation_df
    .orderBy(
        "status",
        "check_name",
    )
)

display(
    validation_df
    .orderBy(
        "status",
        "check_name",
    )
)

In [0]:
# Fail the notebook if any check fails
failed_checks = (
    validation_df
    .filter(
        F.col("status") == "FAIL"
    )
    .count()
)

passed_checks = (
    validation_df
    .filter(
        F.col("status") == "PASS"
    )
    .count()
)

print(
    f"Passed checks: {passed_checks}"
)

print(
    f"Failed checks: {failed_checks}"
)

## Persist validation results

Validation results are retained in the metadata schema so previous pipeline validation runs can be audited.

In [0]:
(
    validation_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        VALIDATION_TABLE
    )
)

print(
    f"Validation results written to: "
    f"{VALIDATION_TABLE}"
)

In [0]:
if failed_checks > 0:

    display(
        validation_df
        .filter(
            F.col("status") == "FAIL"
        )
        .orderBy("check_name")
    )

    raise ValueError(
        f"Gold validation failed: "
        f"{failed_checks} checks failed."
    )


print(
    "Gold layer validation passed."
)

In [0]:
%sql
-- Review validation audit history
SELECT
    validation_run_id,
    status,
    COUNT(*) AS checks,
    MIN(checked_at) AS validated_at
FROM workspace.urbanpulse_meta.gold_validation_results
GROUP BY
    validation_run_id,
    status
ORDER BY validated_at DESC;

In [0]:
%sql
--Review latest failures
WITH latest_run AS (
    SELECT validation_run_id
    FROM workspace.urbanpulse_meta.gold_validation_results
    ORDER BY checked_at DESC
    LIMIT 1
)

SELECT
    r.check_name,
    r.actual_value,
    r.expected_value,
    r.details,
    r.checked_at

FROM workspace.urbanpulse_meta.gold_validation_results r

INNER JOIN latest_run l
    ON r.validation_run_id
       = l.validation_run_id

WHERE r.status = 'FAIL'

ORDER BY r.check_name;

In [0]:
%sql
-- Final Gold inventory
SHOW TABLES
IN workspace.urbanpulse_gold;

In [0]:
%sql
-- Final row-count snapshot
SELECT
    'dim_date' AS table_name,
    COUNT(*) AS rows
FROM workspace.urbanpulse_gold.dim_date

UNION ALL

SELECT
    'dim_line',
    COUNT(*)
FROM workspace.urbanpulse_gold.dim_line

UNION ALL

SELECT
    'dim_station',
    COUNT(*)
FROM workspace.urbanpulse_gold.dim_station

UNION ALL

SELECT
    'bridge_station_line',
    COUNT(*)
FROM workspace.urbanpulse_gold.bridge_station_line

UNION ALL

SELECT
    'fact_line_status',
    COUNT(*)
FROM workspace.urbanpulse_gold.fact_line_status

UNION ALL

SELECT
    'fact_arrival_observation',
    COUNT(*)
FROM workspace.urbanpulse_gold.fact_arrival_observation

UNION ALL

SELECT
    'fact_weather',
    COUNT(*)
FROM workspace.urbanpulse_gold.fact_weather

UNION ALL

SELECT
    'current_line_status',
    COUNT(*)
FROM workspace.urbanpulse_gold.current_line_status

UNION ALL

SELECT
    'station_arrival_summary',
    COUNT(*)
FROM workspace.urbanpulse_gold.station_arrival_summary

UNION ALL

SELECT
    'daily_network_kpis',
    COUNT(*)
FROM workspace.urbanpulse_gold.daily_network_kpis

ORDER BY table_name;